In [ ]:
!pip install git+https://github.com/huggingface/parler-tts.git transformers torch soundfile


In [ ]:
pip install --upgrade protobuf

In [ ]:
from huggingface_hub import login

login(token="your_hf_token_here")

In [ ]:
import json
import torch
import soundfile as sf
from transformers import AutoTokenizer
from parler_tts import ParlerTTSForConditionalGeneration

def generate_parler_audio(json_filepath):
    # 1. Setup Device
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # 2. Load the Gated Indic Parler-TTS Model and Tokenizer
    print("Loading Indic Parler-TTS model and tokenizer...")
    model_id = "ai4bharat/indic-parler-tts"

    try:
        model = ParlerTTSForConditionalGeneration.from_pretrained(model_id).to(device)
        tokenizer = AutoTokenizer.from_pretrained(model_id)
    except Exception as e:
        print("Error: Could not load the model.")
        print("Make sure you have accepted the terms at https://huggingface.co/ai4bharat/indic-parler-tts")
        print("and run 'huggingface-cli login' with a valid token.")
        return

    # 3. Load the Dataset
    print(f"Reading {json_filepath}...")
    with open(json_filepath, "r", encoding="utf-8") as file:
        dataset = json.load(file)

    sentences = dataset.get("sentences", [])

    if not sentences:
        print("Error: Could not find the 'sentences' array in the JSON.")
        return

    # 4. Define the Voice Condition (Caption)
    # The model uses this to determine gender, pitch, speed, and audio quality
    description = "A male speaker delivers a clear and slightly expressive speech at a moderate speed. The recording is of very high quality, with the speaker's voice sounding clear and very close up, with no background noise."

    print("Tokenizing the voice description...")
    input_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)

    # 5. Generate Audio for each sentence
    for item in sentences:
        sentence_id = item["id"]
        text = item["sentence"]

        print(f"Processing Sentence {sentence_id}/{len(sentences)}...")

        # Tokenize the Urdu text
        prompt_input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)

        # Generate the waveform
        with torch.no_grad():
            generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)

        # Extract and Save the Audio
        audio_array = generation.cpu().numpy().squeeze()
        sample_rate = model.config.sampling_rate

        output_filename = f"parler_urdu_sentence_{sentence_id:02d}.wav"
        sf.write(output_filename, audio_array, sample_rate)

    print("\nSuccess! All Parler-TTS audio files have been generated.")

# Run the pipeline
generate_parler_audio("urdu_balanced_set.json")

In [ ]:
import os
import shutil
import glob

# 1. Define folder and zip names specifically for Parler outputs
folder_name = "parler_urdu_audio"
zip_filename = "parler_urdu_dataset"

# Create the directory
os.makedirs(folder_name, exist_ok=True)

# 2. Find and move all Parler .wav files
wav_files = glob.glob("parler_urdu_sentence_*.wav")

if not wav_files:
    print("No Parler .wav files found! Make sure the generation script finished successfully.")
else:
    for file in wav_files:
        shutil.move(file, os.path.join(folder_name, file))

    print(f"Moved {len(wav_files)} audio files into '{folder_name}/'.")

    # 3. Create the ZIP archive
    shutil.make_archive(zip_filename, 'zip', folder_name)
    print(f"Successfully compressed into: {zip_filename}.zip")

    # 4. Trigger the Colab download
    try:
        from google.colab import files
        print("Starting download...")
        files.download(f"{zip_filename}.zip")
    except ImportError:
        print("Not running in Google Colab. The file is ready in your directory.")

In [ ]:
pip install transformers torch jiwer librosa

In [ ]:
import os
import json
import zipfile
import re
import torch
import jiwer
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration

def normalize_urdu_text(text):
    """Strips punctuation and normalizes spaces for Urdu."""
    text = re.sub(r'[۔،؛؟,.;?!]', '', text)
    text = text.replace('\u200c', '').replace('\u200d', '')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def run_parler_evaluation(zip_filepath, json_filepath):
    # 1. Unzip the audio files
    extract_dir = "parler_extracted_eval"
    print(f"Unzipping {zip_filepath}...")
    with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    # 2. Load the Ground Truth JSON
    with open(json_filepath, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    ground_truth_dict = {item["id"]: item["sentence"] for item in dataset["sentences"]}

    # 3. Load Whisper Medium
    print("Loading Whisper Medium ASR model...")
    model_id = "openai/whisper-medium"
    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperForConditionalGeneration.from_pretrained(model_id)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    # 4. Process and Transcribe
    references = []
    hypotheses = []

    wav_files = []
    for root, dirs, files in os.walk(extract_dir):
        for file in files:
            if file.endswith(".wav"):
                wav_files.append(os.path.join(root, file))

    wav_files.sort()

    print("\nStarting Transcription...\n")
    for audio_path in wav_files:
        try:
            # Extracts the numeric ID from 'parler_urdu_sentence_01.wav'
            file_id = int(re.search(r'\d+', os.path.basename(audio_path)).group())
        except AttributeError:
            continue

        if file_id not in ground_truth_dict:
            continue

        original_text = ground_truth_dict[file_id]

        # Load and resample to Whisper's native 16kHz
        audio_array, sr = librosa.load(audio_path, sr=16000)

        inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt").input_features.to(device)
        with torch.no_grad():
            forced_decoder_ids = processor.get_decoder_prompt_ids(language="urdu", task="transcribe")
            predicted_ids = model.generate(inputs, forced_decoder_ids=forced_decoder_ids)

        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        ref_clean = normalize_urdu_text(original_text)
        hyp_clean = normalize_urdu_text(transcription)

        references.append(ref_clean)
        hypotheses.append(hyp_clean)

        print(f"File: {os.path.basename(audio_path)}")
        print(f"  Ref: {ref_clean}")
        print(f"  Hyp: {hyp_clean}")
        print("-" * 50)

    # 5. Calculate Metrics
    if not references:
        print("Error: No valid files were processed.")
        return

    wer = jiwer.wer(references, hypotheses)
    cer = jiwer.cer(references, hypotheses)

    print("\n" + "="*45)
    print("FINAL EVALUATION METRICS (Indic Parler-TTS)")
    print("="*45)
    print(f"Word Error Rate (WER):      {wer:.4f} ( {wer*100:.2f}% )")
    print(f"Character Error Rate (CER): {cer:.4f} ( {cer*100:.2f}% )")
    print("="*45)

# RUN THE FUNCTION
run_parler_evaluation("parler_urdu_dataset.zip", "urdu_balanced_set.json")